# Phase 7: Growth-Dynamics Modelling

Evaluate whether next-day density, absolute change, relative growth, or log-specific growth are useful formulations for experimental Chlorella data. This phase uses descriptive analysis and simple grouped baselines only. It does not train deep-learning models or claim Liquid Tree deployment readiness.

## 1. Load the processed data and construct adjacent-day pairs

In [ ]:
from pathlib import Path
import importlib
import sys

import pandas as pd

INPUT_PATH = Path("data/processed/hiltunen_growth_long.csv")
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next((path for path in search_locations if (path / INPUT_PATH).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not find {INPUT_PATH} from this directory or its parents.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.models.growth_dynamics_analysis as growth_analysis
growth_analysis = importlib.reload(growth_analysis)
analysis = growth_analysis.run_growth_dynamics_analysis(PROJECT_ROOT)
pairs = analysis["pairs"]
pair_audit = analysis["pair_audit"]
distributions = analysis["distributions"]
growth_flags = analysis["growth_flags"]
flagged_rows = analysis["flagged_rows"]
experiment_summary = analysis["experiments"]
condition_summary = analysis["conditions"]
correlations = analysis["correlations"]
model_results = analysis["model_results"]
availability = analysis["availability"]

## 2. Consecutive-pair verification
Pairs are created within `Exp` + `Funnel#` only when `Next_Day == Day + 1` and both density endpoints are observed. No density values are imputed and no non-consecutive observations are linked.

In [ ]:
print({key: value for key, value in pair_audit.items() if key != "day_counts"})
day_counts = pair_audit["day_counts"].rename("Calculated_Valid_Pairs").rename_axis("Day").reset_index()
phase4_counts = pd.read_csv(PROJECT_ROOT / "data/processed/hiltunen_next_day_pairs_by_day.csv")
day_counts = day_counts.merge(phase4_counts[["Day", "Valid_Next_Day_Pairs"]], on="Day", validate="one_to_one")
display(day_counts)
assert pair_audit["valid_pairs"] == 689
assert pair_audit["matches_phase4"]
print("Phase 4 comparison passed: 689 valid adjacent-day density pairs.")

## 3. Growth-target definitions, distributions, and unusual observations

- `Absolute_Growth = Next_Cell_Density - Current_Cell_Density` ($10^6$ cells/mL per day for the one-day interval).
- `Relative_Growth = Absolute_Growth / Current_Cell_Density` (fraction per day).
- `Specific_Growth_Rate = ln(Next_Cell_Density / Current_Cell_Density)` (day$^{-1}$ for a one-day interval).

Relative and specific growth are calculated only for positive endpoints. IQR fences and a descriptive near-zero threshold of 0.1 density units flag observations for review only; nothing is removed.

In [ ]:
display(distributions)
display(growth_flags)
print(f"IQR-flagged target rows (including pairs flagged on multiple targets): {len(flagged_rows)}")
display(
    pairs.assign(Absolute_Growth_Magnitude=pairs["Absolute_Growth"].abs())
    .nlargest(8, "Absolute_Growth_Magnitude")[
        ["Exp", "Funnel#", "Day", "Medium", "CO2", "Light",
         "Current_Cell_Density", "Next_Cell_Density", "Absolute_Growth",
         "Relative_Growth", "Specific_Growth_Rate"]
    ]
)
print("Flagged observations are retained and shown with their culture and condition context; unusual values are not presumed erroneous.")

## 4. Growth trajectories and dynamics by day

In [ ]:
from IPython.display import Image, display

for filename in [
    "growth_cell_density_trajectories.png",
    "growth_absolute_by_day.png",
    "growth_relative_by_day.png",
    "growth_specific_by_day.png",
]:
    print(filename)
    display(Image(filename=str(analysis["figure_directory"] / filename)))

## 5. Compare experiments

In [ ]:
display(experiment_summary)

## 6. Descriptive condition analysis
CO2, Light, and Medium are summarized as categorical labels exactly as stored. These grouped summaries are not causal comparisons and do not identify an optimum.

In [ ]:
display(condition_summary)

## 7. Associations
Spearman correlations are descriptive. Growth formulas use current density algebraically, so associations between current density and change/rates can reflect mathematical coupling; correlation does not establish causation.

In [ ]:
display(correlations)

## 8. Leakage-safe baseline growth models
A fold-specific training-mean baseline, Linear Regression, and the Phase 5 Random Forest configuration are evaluated for each target. Category encoding is fit inside each training fold. Identifiers are groups only, not predictors. MAE/RMSE values across targets have different units/scales and should not be directly ranked.

In [ ]:
summary_results = (
    model_results.groupby(["Target", "Model", "Evaluation_Protocol"], sort=False)
    .first()
    .reset_index()
)
display(summary_results[["Target", "Model", "Evaluation_Protocol", "Total_Sample_Count", "MAE_Mean", "MAE_SD", "RMSE_Mean", "RMSE_SD", "R2_Mean", "R2_SD"]])
print("Fold-level LOEO rows, with held-out experiment identity:")
display(model_results.loc[model_results["Evaluation_Protocol"].eq("LeaveOneExperimentOut"), ["Target", "Model", "Fold", "Sample_Count", "MAE", "RMSE", "R2"]])

## 9. Environmental-variable availability
The long modelling table and raw source workbook are distinguished here. Workbook descriptions of categorical treatments are context only, not fabricated per-observation numerical predictors.

In [ ]:
display(availability)

## 10. Target formulation interpretation and limitations
Absolute next-day density remains directly interpretable for short-horizon forecasting. Absolute change expresses the density increment in source units; relative and specific rates normalize change by current density, but depend on a denominator and have distinct scales. The current dataset supports exploratory comparisons of all three targets, not a single target winner.

The four experiments and 61 cultures provide limited independent validation, and only four valid pairs remain on current days 13-15. Treatment summaries can be entangled with experiment design. The long table lacks numeric per-culture CO2/light measurements and temperature; nutrient/DIC fields in the wider workbook are not in the long modelling table, with end-of-experiment measurements unavailable at arbitrary current day. This is experimental Chlorella data, not actual Liquid Tree sensor data.

Growth dynamics are scientifically justified for continued transparent exploratory modelling and may eventually support forecasting, utilization accounting, or harvest analysis after validation. For a predictive/control target decision, retain absolute density and growth change as parallel candidates while collecting additional independent experiments and actual Liquid Tree data. The evidence does not justify ANN/LSTM/GRU yet. No Phase 8 work is included.